<a href="https://colab.research.google.com/github/Mostafa-Seyedi/MaskArchitectureAnomaly_CourseProject/blob/main/step_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup
Mount Google Drive, clone repository, install dependencies and extract anomaly validation datasets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/AlessandroMarinai/MaskArchitectureAnomaly_CourseProject
%cd /content/MaskArchitectureAnomaly_CourseProject
!pip install -q ood-metrics scikit-learn numpy==1.26.4

!unzip -q /content/drive/MyDrive/MLDL_NewVersionProject/Validation_Dataset.zip -d /content/anomaly_datasets

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 131 (delta 22), reused 19 (delta 19), pack-reused 75 (from 1)
Receiving objects: 100% (131/131), 26.88 MiB | 14.84 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/MaskArchitectureAnomaly_CourseProject
replace /content/anomaly_datasets/Validation_Dataset/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


## 2. Imports
Core libraries for anomaly segmentation evaluation. `ood-metrics` provides FPR@95TPR calculation. `sklearn` provides AuPRC (Average Precision).

In [ ]:
!pip install "numpy<2.0"

In [ ]:
import os
import sys
import glob
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr
import matplotlib.pyplot as plt

sys.path.insert(0, '/content/MaskArchitectureAnomaly_CourseProject/eval')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

input_transform = Compose([
    Resize((512, 1024), Image.BILINEAR),
    ToTensor(),
])
target_transform = Compose([
    Resize((512, 1024), Image.NEAREST),
])

print("✅ Imports done")

Device: cuda
✅ Imports done


## 3. Load ERFNet Model
ERFNet is a lightweight CNN for real-time semantic segmentation trained on Cityscapes (19+1 classes). We use the pretrained weights from the repository as the pixel-based baseline model.

In [ ]:
from erfnet import ERFNet

NUM_CLASSES = 20
model_erfnet = ERFNet(NUM_CLASSES)
model_erfnet = torch.nn.DataParallel(model_erfnet).to(DEVICE)

WEIGHTS_PATH = '/content/MaskArchitectureAnomaly_CourseProject/trained_models/erfnet_pretrained.pth'

def load_my_state_dict(model, state_dict):
    own_state = model.state_dict()
    for name, param in state_dict.items():
        if name not in own_state:
            if name.startswith("module."):
                own_state[name.split("module.")[-1]].copy_(param)
        else:
            own_state[name].copy_(param)
    return model

model_erfnet = load_my_state_dict(
    model_erfnet,
    torch.load(WEIGHTS_PATH, map_location=DEVICE)
)
model_erfnet.eval()
print("✅ ERFNet loaded")

✅ ERFNet loaded


## 4. Dataset Paths & Helper Functions
Define paths for the 5 anomaly segmentation benchmarks and helper functions for loading ground truth masks and computing metrics.

Datasets:
- **SMIYC RA-21**: RoadAnomaly21 — animals and objects on road
- **SMIYC RO-21**: RoadObsticle21 — road obstacles
- **FS L&F**: LostAndFound — small obstacles on road  
- **FS Static**: Fishyscapes Static — pasted anomalous objects
- **Road Anomaly**: General road anomalies

In [ ]:
BASE = '/content/anomaly_datasets/Validation_Dataset'

DATASETS = {
    'SMIYC_RA21':  f'{BASE}/RoadAnomaly21/images/*.png',
    'SMIYC_RO21':  f'{BASE}/RoadObsticle21/images/*.webp',
    'FS_LnF':      f'{BASE}/FS_LostFound_full/images/*.png',
    'FS_Static':   f'{BASE}/fs_static/images/*.jpg',
    'RoadAnomaly': f'{BASE}/RoadAnomaly/images/*.jpg',
}

def get_gt_path(img_path):
    """Exactly as in TA's evalAnomaly.py"""
    pathGT = img_path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT and "RoadAnomaly21" not in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    return pathGT

def load_gt(pathGT):
    """Exactly as in TA's evalAnomaly.py"""
    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)

    if "RoadAnomaly" in pathGT:
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)

    if "LostAndFound" in pathGT or "FS_LostFound" in pathGT:
        # This dataset version has 0=bg, 1=anomaly, 255=ignore
        # No remapping needed
        pass

    if "Streethazard" in pathGT:
        ood_gts = np.where((ood_gts == 14), 255, ood_gts)
        ood_gts = np.where((ood_gts < 20), 0, ood_gts)
        ood_gts = np.where((ood_gts == 255), 1, ood_gts)

    return ood_gts

def compute_metrics(anomaly_score_list, ood_gts_list):
    """Exactly as in TA's evalAnomaly.py"""
    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_mask = (ood_gts == 1)
    ind_mask = (ood_gts == 0)

    ood_out = anomaly_scores[ood_mask]
    ind_out = anomaly_scores[ind_mask]

    val_out   = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))

    auprc = average_precision_score(val_label, val_out) * 100
    fpr95 = fpr_at_95_tpr(val_out, val_label) * 100
    return auprc, fpr95

# Verify all datasets
print("Datasets:")
for name, pattern in DATASETS.items():
    files = glob.glob(pattern)
    print(f"  {name}: {len(files)} images")

# Quick GT check for FS_LnF
path = sorted(glob.glob(f'{BASE}/FS_LostFound_full/images/*.png'))[0]
gt = load_gt(get_gt_path(path))
print(f"\nFS_LnF GT check — unique values: {np.unique(gt)}")
print(f"Contains anomaly: {1 in np.unique(gt)} ✅" if 1 in np.unique(gt) else "No anomaly found ❌")

Datasets:
  SMIYC_RA21: 10 images
  SMIYC_RO21: 30 images
  FS_LnF: 100 images
  FS_Static: 30 images
  RoadAnomaly: 60 images

FS_LnF GT check — unique values: [  0   1 255]
Contains anomaly: True ✅


## 5. ERFNet Pixel-based Baselines
Apply three post-hoc anomaly scoring methods on ERFNet:

- **MSP** (Maximum Softmax Probability): anomaly score = 1 - max(softmax). Low confidence = anomaly.
- **MaxLogit**: anomaly score = -max(logit). Uses raw logits before softmax, more calibrated.
- **MaxEntropy**: anomaly score = entropy of softmax distribution. High entropy = uncertain = anomaly.

In [ ]:
def run_all_methods_erfnet(model, dataset_pattern, dataset_name):
    msp_s, ml_s, me_s, gts = [], [], [], []
    paths = sorted(glob.glob(dataset_pattern))
    if not paths:
        return None
    print(f"  {len(paths)} images")

    for path in paths:
        img = input_transform(
            Image.open(path).convert('RGB')
        ).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            logits = model(img)  # 1x20xHxW

        logits_np = logits.squeeze(0).cpu().numpy()
        softmax   = F.softmax(logits.squeeze(0), dim=0).cpu().numpy()

        msp_score = 1.0 - np.max(softmax, axis=0)
        ml_score  = -np.max(logits_np, axis=0)
        me_score  = -np.sum(softmax * np.log(softmax + 1e-8), axis=0)

        pathGT = get_gt_path(path)
        try:
            gt = load_gt(pathGT)
        except:
            continue
        if 1 not in np.unique(gt):
            continue

        msp_s.append(msp_score)
        ml_s.append(ml_score)
        me_s.append(me_score)
        gts.append(gt)
        torch.cuda.empty_cache()

    if not gts:
        return None
    return {
        'MSP':        compute_metrics(msp_s, gts),
        'MaxLogit':   compute_metrics(ml_s,  gts),
        'MaxEntropy': compute_metrics(me_s,  gts),
    }

print("=== ERFNet Evaluation ===")
erfnet_results = {}
for name, pattern in DATASETS.items():
    print(f"\n{name}:")
    r = run_all_methods_erfnet(model_erfnet, pattern, name)
    if r:
        erfnet_results[name] = r
        for m, (a, f) in r.items():
            print(f"  {m:<12}: AuPRC={a:.2f}%  FPR95={f:.2f}%")
    else:
        print("  No valid images")
print("\n✅ ERFNet done")

=== ERFNet Evaluation ===

SMIYC_RA21:
  10 images
  MSP         : AuPRC=29.10%  FPR95=62.55%
  MaxLogit    : AuPRC=38.32%  FPR95=59.34%
  MaxEntropy  : AuPRC=30.97%  FPR95=62.66%

SMIYC_RO21:
  30 images
  MSP         : AuPRC=2.71%  FPR95=65.22%
  MaxLogit    : AuPRC=4.63%  FPR95=48.44%
  MaxEntropy  : AuPRC=3.04%  FPR95=65.91%

FS_LnF:
  100 images
  MSP         : AuPRC=1.75%  FPR95=50.59%
  MaxLogit    : AuPRC=3.30%  FPR95=45.49%
  MaxEntropy  : AuPRC=2.58%  FPR95=50.16%

FS_Static:
  30 images
  MSP         : AuPRC=7.47%  FPR95=41.84%
  MaxLogit    : AuPRC=9.50%  FPR95=40.30%
  MaxEntropy  : AuPRC=8.84%  FPR95=41.55%

RoadAnomaly:
  60 images
  MSP         : AuPRC=12.42%  FPR95=82.58%
  MaxLogit    : AuPRC=15.58%  FPR95=73.25%
  MaxEntropy  : AuPRC=12.67%  FPR95=82.75%

✅ ERFNet done


## 6. Load EoMT Model
Load EoMT-Cityscapes for mask-based anomaly evaluation. The model uses a ViT-Base backbone with DINOv2 pretraining.

In [ ]:
import wandb, yaml, importlib, warnings
from lightning import seed_everything

seed_everything(0, verbose=False)
os.chdir('/content/MaskArchitectureAnomaly_CourseProject/eomt')
sys.path.insert(0, '/content/MaskArchitectureAnomaly_CourseProject/eomt')

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml") as f:
    config_city = yaml.safe_load(f)

def build_eomt(checkpoint_path, num_classes=19):
    encoder_cfg = config_city["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_mod, enc_cls = encoder_cfg["class_path"].rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(enc_mod), enc_cls)
    enc_init = {k: v for k, v in encoder_cfg.get("init_args", {}).items()}
    enc_init["img_size"]   = (1024, 1024)  # match backbone
    enc_init["patch_size"] = 16
    encoder = encoder_cls(**enc_init)

    net_cfg = config_city["model"]["init_args"]["network"]
    net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
    network_cls = getattr(importlib.import_module(net_mod), net_cls)
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k != "encoder"}
    network = network_cls(
        masked_attn_enabled=False,
        num_classes=num_classes,
        encoder=encoder, **net_kwargs
    )

    warnings.filterwarnings("ignore", message=r".*Attribute 'network'.*")
    lit_mod, lit_cls_name = config_city["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_mod), lit_cls_name)
    m_kwargs = {k: v for k, v in config_city["model"]["init_args"].items() if k != "network"}

    model = lit_cls(
        img_size=(1024, 1024), num_classes=num_classes,
        network=network, **m_kwargs
    ).eval().to(DEVICE)
    model.img_size = (1024, 1024)  # must match encoder

    state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    if 'model_state_dict' in state:
        state = state['model_state_dict']
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"  Missing: {len(missing)} | Unexpected: {len(unexpected)}")
    return model

CKPT_CITY = '/content/drive/MyDrive/MLDL_NewVersionProject/eomt_cityscapes.bin'
print("Loading EoMT-Cityscapes...")
model_eomt = build_eomt(CKPT_CITY, num_classes=19)
print("✅ EoMT-Cityscapes loaded")

Loading EoMT-Cityscapes...
  Missing: 0 | Unexpected: 0
✅ EoMT-Cityscapes loaded


## 7. EoMT Mask-based Baselines
Apply post-hoc methods on EoMT. Since EoMT is a mask architecture, the output is converted to per-pixel logits using the semantic inference pipeline from Step 4.

In [ ]:
def run_all_methods_eomt(model, dataset_pattern, dataset_name):
    msp_s, ml_s, me_s, gts = [], [], [], []
    paths = sorted(glob.glob(dataset_pattern))
    if not paths:
        return None
    print(f"  {len(paths)} images")

    for path in paths:
        image = input_transform(Image.open(path).convert('RGB'))
        image_uint8 = (image * 255).byte()

        with torch.no_grad(), torch.amp.autocast('cuda'):
            imgs     = [image_uint8.to(DEVICE)]
            img_sizes = [image_uint8.shape[-2:]]
            crops, origins = model.window_imgs_semantic(imgs)
            mask_l, class_l = model(crops)
            mask_logits = F.interpolate(
                mask_l[-1], model.img_size, mode='bilinear')
            per_pixel = model.to_per_pixel_logits_semantic(
                mask_logits, class_l[-1])
            logits_full = model.revert_window_logits_semantic(
                per_pixel, origins, img_sizes)[0]

        logits_r = F.interpolate(
            logits_full.unsqueeze(0), size=(512, 1024), mode='bilinear'
        ).squeeze(0).cpu().float()

        logits_np = logits_r.numpy()
        softmax   = F.softmax(logits_r, dim=0).numpy()

        msp_score = 1.0 - np.max(softmax, axis=0)
        ml_score  = -np.max(logits_np, axis=0)
        me_score  = -np.sum(softmax * np.log(softmax + 1e-8), axis=0)

        pathGT = get_gt_path(path)
        try:
            gt = load_gt(pathGT)
        except:
            continue
        if 1 not in np.unique(gt):
            continue

        msp_s.append(msp_score)
        ml_s.append(ml_score)
        me_s.append(me_score)
        gts.append(gt)
        torch.cuda.empty_cache()

    if not gts:
        return None
    return {
        'MSP':        compute_metrics(msp_s, gts),
        'MaxLogit':   compute_metrics(ml_s,  gts),
        'MaxEntropy': compute_metrics(me_s,  gts),
    }

print("=== EoMT-Cityscapes Evaluation ===")
eomt_results = {}
for name, pattern in DATASETS.items():
    print(f"\n{name}:")
    r = run_all_methods_eomt(model_eomt, pattern, name)
    if r:
        eomt_results[name] = r
        for m, (a, f) in r.items():
            print(f"  {m:<12}: AuPRC={a:.2f}%  FPR95={f:.2f}%")
    else:
        print("  No valid images")
print("\n✅ EoMT done")

=== EoMT-Cityscapes Evaluation ===

SMIYC_RA21:
  10 images
  MSP         : AuPRC=69.60%  FPR95=14.10%
  MaxLogit    : AuPRC=69.39%  FPR95=14.38%
  MaxEntropy  : AuPRC=69.97%  FPR95=14.35%

SMIYC_RO21:
  30 images
  MSP         : AuPRC=84.27%  FPR95=4.49%
  MaxLogit    : AuPRC=84.11%  FPR95=4.61%
  MaxEntropy  : AuPRC=84.30%  FPR95=4.62%

FS_LnF:
  100 images
  MSP         : AuPRC=16.85%  FPR95=10.17%
  MaxLogit    : AuPRC=16.79%  FPR95=9.94%
  MaxEntropy  : AuPRC=19.53%  FPR95=9.93%

FS_Static:
  30 images
  MSP         : AuPRC=62.36%  FPR95=39.03%
  MaxLogit    : AuPRC=62.49%  FPR95=39.90%
  MaxEntropy  : AuPRC=59.92%  FPR95=40.44%

RoadAnomaly:
  60 images
  MSP         : AuPRC=67.97%  FPR95=25.53%
  MaxLogit    : AuPRC=67.42%  FPR95=25.53%
  MaxEntropy  : AuPRC=69.87%  FPR95=25.13%

✅ EoMT done


## 8. Results Table
Summary of all baseline results for ERFNET and EoMT across all 5 anomaly segmentation benchmarks.
Higher AuPRC = better. Lower FPR95 = better.

In [ ]:
import pandas as pd

DS = ['SMIYC_RA21','SMIYC_RO21','FS_LnF','FS_Static','RoadAnomaly']
DS_LABEL = ['SMIYC RA-21','SMIYC RO-21','FS L&F','FS Static','Road Anomaly']
METHODS = ['MSP','MaxLogit','MaxEntropy']

def g(res, ds, method, idx):
    try:    return round(res[ds][method][idx], 1)
    except: return '-'

rows = []
for method in METHODS:
    row = {'Model':'ERFNET', 'mIoU':'----' if method=='MSP' else '', 'Method': method}
    for ds, dl in zip(DS, DS_LABEL):
        row[f'{dl} AuPRC'] = g(erfnet_results, ds, method, 0)
        row[f'{dl} FPR95'] = g(erfnet_results, ds, method, 1)
    rows.append(row)

for method in METHODS:
    row = {'Model':'EoMT', 'mIoU':'----' if method=='MSP' else '', 'Method': method}
    for ds, dl in zip(DS, DS_LABEL):
        row[f'{dl} AuPRC'] = g(eomt_results, ds, method, 0)
        row[f'{dl} FPR95'] = g(eomt_results, ds, method, 1)
    rows.append(row)

df = pd.DataFrame(rows)
df = df.set_index(['Model','mIoU','Method'])

# Display nicely
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(df.to_string())

# Also save as CSV
df.to_csv('/content/drive/MyDrive/MLDL_NewVersionProject/step7_results.csv')
print("\n✅ Saved to Drive as step7_results.csv")

                        SMIYC RA-21 AuPRC  SMIYC RA-21 FPR95  SMIYC RO-21 AuPRC  SMIYC RO-21 FPR95  FS L&F AuPRC  FS L&F FPR95  FS Static AuPRC  FS Static FPR95  Road Anomaly AuPRC  Road Anomaly FPR95
Model  mIoU Method                                                                                                                                                                                      
ERFNET ---- MSP                      29.1               62.5                2.7               65.2           1.7          50.6              7.5             41.8                12.4                82.6
            MaxLogit                 38.3               59.3                4.6               48.4           3.3          45.5              9.5             40.3                15.6                73.2
            MaxEntropy               31.0               62.7                3.0               65.9           2.6          50.2              8.8             41.5                12.7                